# Core Panel Exploration

Use this notebook after running `python -m src.pipelines.build_core_panel` from the repo root.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from src import config

PANEL_MODE = "live"  # switch to "live" after running the live pipeline

core_panel = pd.read_csv(config.core_panel_csv(PANEL_MODE), parse_dates=["Date", "Inception"])
macro_factors = pd.read_csv(config.macro_factors_weekly_csv(PANEL_MODE), parse_dates=["Date"])
weekly_returns = pd.read_csv(config.weekly_returns_long_csv(PANEL_MODE), parse_dates=["Date"])

print(f"Loaded {PANEL_MODE} panel from {config.processed_dir_for_mode(PANEL_MODE)}")
core_panel.head()

Loaded live panel from C:\Users\yerr\Desktop\fixed-income-etf-macro-risk\data\processed\live


,Date,Symbol,Return,ANFCI,BAMLC0A0CM,DGS10,T10Y2Y,T5YIE,KCPRU,VIX,...,stress_index,high_stress,vol_12w,var_12w_90,downside_vol_12w,maxdd_12w,es_12w_90,fwd_ret_4w,fwd_maxdd_12w,fwd_vol_12w
0,2020-09-18,AAA,-0.002397,-0.689,1.35,0.70,0.56,1.58,0.253,25.830000,...,-0.305741,0,NaN,NaN,NaN,NaN,NaN,-0.002083,-0.002883,0.001823
1,2020-09-25,AAA,0.000200,-0.679,1.46,0.66,0.54,1.43,0.237,26.379999,...,0.525941,0,NaN,NaN,NaN,NaN,NaN,-0.000480,-0.001884,0.001822
2,2020-10-02,AAA,-0.002082,-0.648,1.43,0.70,0.57,1.48,0.224,27.629999,...,0.067168,0,NaN,NaN,NaN,NaN,NaN,0.001405,-0.001884,0.001630
3,2020-10-09,AAA,-0.000803,-0.604,1.35,0.79,0.63,1.56,0.249,25.000000,...,0.013128,0,NaN,NaN,NaN,NaN,NaN,0.000521,-0.001884,0.001572
4,2020-10-16,AAA,0.000603,-0.570,1.33,0.76,0.62,1.53,0.246,27.410000,...,0.329534,0,NaN,NaN,NaN,NaN,NaN,0.004941,-0.001884,0.001588


In [2]:
core_panel.info()

<class 'pandas.DataFrame'>
RangeIndex: 159291 entries, 0 to 159290
Data columns (total 49 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Date                   159291 non-null  datetime64[us]
 1   Symbol                 159291 non-null  str           
 2   Return                 159291 non-null  float64       
 3   ANFCI                  159291 non-null  float64       
 4   BAMLC0A0CM             159291 non-null  float64       
 5   DGS10                  159291 non-null  float64       
 6   T10Y2Y                 159291 non-null  float64       
 7   T5YIE                  159291 non-null  float64       
 8   KCPRU                  159291 non-null  float64       
 9   VIX                    159291 non-null  float64       
 10  MOVE                   159291 non-null  float64       
 11  GPR                    159291 non-null  float64       
 12  d_ANFCI                159291 non-null  float64       


In [3]:
core_panel[["Return", "RF_w", "RET_XS", "age_years", "Assets_clean", "ER_clean"]].describe()

,Return,RF_w,RET_XS,age_years,Assets_clean,ER_clean
count,159291.000000,159291.000000,159291.000000,159291.000000,1.592910e+05,159291.000000
mean,0.000491,0.000488,0.000003,7.867808,6.211225e+09,0.003264
std,0.012516,0.000366,0.012507,4.904621,1.552216e+10,0.003892
min,-0.243252,0.000002,-0.243262,0.019165,5.889600e+06,0.000000
25%,-0.002456,0.000100,-0.002960,3.898700,2.406760e+08,0.001200
50%,0.000647,0.000450,0.000164,7.236140,1.070970e+09,0.002200
75%,0.003790,0.000813,0.003315,11.411362,4.550980e+09,0.004100
max,0.252916,0.001031,0.252830,23.986311,1.496970e+11,0.047200


In [4]:
core_panel.groupby("ETF Database Category")["RET_XS"].agg(["count", "mean", "std"]).sort_values("count", ascending=False).head(20)

,count,mean,std
ETF Database Category,,,
Total Bond Market,36438,-0.000089,0.007558
Corporate Bonds,30680,0.000042,0.010417
High Yield Bonds,22148,0.000455,0.012719
National Munis,18127,-0.000090,0.009574
Government Bonds,16465,-0.000366,0.011584
Inflation-Protected Bonds,7867,0.000034,0.009886
Emerging Markets Bonds,6722,0.000079,0.013387
Mortgage Backed Securities,6520,-0.000177,0.006897
Inverse Bonds,4136,0.000460,0.035360


## Core Transform Validation Checks

These lightweight checks cover the highest-risk panel transforms before relying on the exploratory summaries.

In [5]:
from src.data.prices import build_weekly_returns
from src.features.forward_outcomes import add_forward_outcomes
from src.features.structural import parse_assets, parse_expense_ratio
from src.features.category import CATEGORY_MAP

# Missing weekly prices should remain missing, not become zero returns.
_sparse_prices = pd.DataFrame(
    {
        "ETF_A": [100.0, np.nan, 110.0],
        "ETF_B": [np.nan, 50.0, 55.0],
    },
    index=pd.date_range("2024-01-05", periods=3, freq="W-FRI"),
)
_sparse_returns = build_weekly_returns(_sparse_prices)

_sparse_returns_check = _sparse_returns.reindex(_sparse_prices.index)

assert pd.isna(_sparse_returns_check.loc["2024-01-12", "ETF_A"])
assert pd.isna(_sparse_returns_check.loc["2024-01-19", "ETF_A"])
assert np.isclose(_sparse_returns_check.loc["2024-01-19", "ETF_B"], 0.10)

_sparse_returns

,ETF_A,ETF_B
2024-01-19,NaN,0.1


In [6]:
# Forward outcomes should use future returns only: t+1 ... t+k.
_toy_panel = pd.DataFrame(
    {
        "Symbol": ["AAA"] * 6,
        "Date": pd.date_range("2024-01-05", periods=6, freq="W-FRI"),
        "Return": [0.01, 0.02, -0.01, 0.03, 0.04, -0.02],
    }
)
_toy_outcomes = add_forward_outcomes(
    _toy_panel,
    fwd_short=2,
    fwd_long=3,
    min_periods_long=3,
)

_expected_fwd_ret_2w_at_t0 = (1 + 0.02) * (1 - 0.01) - 1
_expected_fwd_ret_2w_at_t1 = (1 - 0.01) * (1 + 0.03) - 1

assert np.isclose(_toy_outcomes.loc[0, "fwd_ret_4w"], _expected_fwd_ret_2w_at_t0)
assert np.isclose(_toy_outcomes.loc[1, "fwd_ret_4w"], _expected_fwd_ret_2w_at_t1)
assert pd.isna(_toy_outcomes.loc[4, "fwd_ret_4w"])
assert pd.isna(_toy_outcomes.loc[5, "fwd_ret_4w"])

_toy_outcomes

,Symbol,Date,Return,fwd_ret_4w,fwd_maxdd_12w,fwd_vol_12w
0,AAA,2024-01-05,0.01,0.0098,-0.01,0.020817
1,AAA,2024-01-12,0.02,0.0197,0.00,0.026458
2,AAA,2024-01-19,-0.01,0.0712,-0.02,0.032146
3,AAA,2024-01-26,0.03,0.0192,NaN,NaN
4,AAA,2024-02-02,0.04,NaN,NaN,NaN
5,AAA,2024-02-09,-0.02,NaN,NaN,NaN


In [7]:
# Structural parsers should handle common ETFDB formats and missing markers.
assert parse_assets("$1.2B") == 1_200_000_000
assert parse_assets("42,609,200") == 42_609_200
assert parse_assets("--") != parse_assets("--")
assert parse_expense_ratio("0.19%") == 0.0019
assert parse_expense_ratio("-") != parse_expense_ratio("-")

pd.DataFrame(
    {
        "input": ["$1.2B", "42,609,200", "--", "0.19%", "-"],
        "parsed": [
            parse_assets("$1.2B"),
            parse_assets("42,609,200"),
            parse_assets("--"),
            parse_expense_ratio("0.19%"),
            parse_expense_ratio("-"),
        ],
    }
)

,input,parsed
0,$1.2B,1.200000e+09
1,"42,609,200",4.260920e+07
2,--,NaN
3,0.19%,1.900000e-03
4,-,NaN


In [8]:
# Current panel categories should all map into research buckets.
_current_categories = set(core_panel["ETF Database Category"].dropna().unique())
_unmapped_categories = sorted(_current_categories - set(CATEGORY_MAP))

assert not _unmapped_categories, _unmapped_categories

core_panel.groupby("category_bucket")["Symbol"].nunique().sort_values(ascending=False)

category_bucket
Core / Aggregate / Intermediate    83
Investment Grade Corporate         69
High Yield                         49
Muni                               47
Treasury / Government              37
TIPS / Inflation-Linked            16
Mortgage / Securitized             15
EM Debt                            14
Other                              13
DM Debt                             5
Preferred / Hybrid                  4
Short Duration / Cash-like          1
Name: Symbol, dtype: int64